In [ ]:
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import distinctipy
import pickle
from collections import Counter
from tqdm import tqdm 
import cv2
from pathlib import Path
from scipy.ndimage import binary_dilation
from joblib import Parallel, delayed
from tqdm_joblib import tqdm_joblib
import tifffile as tf

In [ ]:
import distinctipy
def create_color_dict(labels_unique, excluded_colors=None):
    color_arr = distinctipy.get_colors(len(labels_unique), exclude_colors=excluded_colors)
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
radius = 100

In [ ]:
key = '/Multiplex 6/MP3-058-2'

In [ ]:
cell_clustering_base_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering' + key

In [ ]:
color_deconvolution_path = 'Y:/coskun-lab/Aditi/01/Results/color_deconvolution' + key 

In [ ]:
pixel_clustering_path = 'Y:/coskun-lab/Aditi/01/Results/pixel_clustering' + key + 
cell_pixel_barcode_path = 'Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key + '/radius100'

In [ ]:
with open(cell_clustering_base_path + '/1x/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f:
    cell_segmentation_mask = pickle.load(f)

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_base_path + '/1x/cell_cluster_location_merged_unclassified_within_contour.csv').drop(columns=['Unnamed: 0', 'index', 'index.1'])

In [ ]:
with open(cell_clustering_base_path + '/1x/cell_type_color_dict.pickle', 'rb') as f: 
    cell_type_color_dict = pickle.load(f)

In [ ]:
trichrome_blue = tf.imread(color_deconvolution_path + '/trichrome_blue.tif')

In [ ]:
from scipy.ndimage import distance_transform_edt
def compute_intensity_outside_cell(cell_idx):
    ys, xs = np.where(cell_segmentation_mask == cell_idx)
    y_min = max(ys.min() - radius, 0)
    y_max = min(ys.max() + radius + 1, cell_segmentation_mask.shape[0])
    x_min = max(xs.min() - radius, 0)
    x_max = min(xs.max() + radius + 1, cell_segmentation_mask.shape[1])
    
    cell_block = (cell_segmentation_mask[y_min:y_max, x_min:x_max] == cell_idx)
    label_block = trichrome_blue[y_min:y_max, x_min:x_max]
    dist = distance_transform_edt(~cell_block) #distance_transform_cdt(~cell_block, metric='chessboard')
    ring_mask = (dist <= radius) & (~cell_block)
    
    return np.mean(label_block[ring_mask])


In [ ]:
from scipy.ndimage import distance_transform_cdt

def compute_intensity_weighted_fibrosis_fraction(cell_idx, max_intensity=1):
    
    ys, xs = np.where(cell_segmentation_mask == cell_idx)
    y_min = max(ys.min() - radius, 0)
    y_max = min(ys.max() + radius + 1, cell_segmentation_mask.shape[0])
    x_min = max(xs.min() - radius, 0)
    x_max = min(xs.max() + radius + 1, cell_segmentation_mask.shape[1])
    cell_block = (cell_segmentation_mask[y_min:y_max, x_min:x_max] == cell_idx)
    label_block = trichrome_blue[y_min:y_max, x_min:x_max]
    dist = distance_transform_edt(~cell_block) #distance_transform_cdt(~cell_block, metric='chessboard')
    ring_mask = (dist <= radius) & (~cell_block)
    fibrosis_intensities = label_block[ring_mask]

    total_intensity = np.sum(fibrosis_intensities)
    max_possible_intensity = max_intensity * np.sum(ring_mask)
 
    intensity_weighted_fraction = total_intensity / max_possible_intensity if max_possible_intensity > 0 else 0
    
    return intensity_weighted_fraction


In [ ]:
cell_id_list = list(cell_cluster_location_df['Cell ID'])

In [ ]:
with tqdm_joblib(tqdm(desc="Processing cells", total=len(cell_id_list))) as progress_bar:
    fibrosis_outside_cells = Parallel(n_jobs=-1)(delayed(compute_intensity_weighted_fibrosis_fraction)(idx) for idx in cell_id_list)

In [ ]:
with open('Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key + '/radius100' + '/weighted_fibrosis_outside_cells.pickle', 'wb') as f: 
    pickle.dump(fibrosis_outside_cells, f)

In [ ]:
with tqdm_joblib(tqdm(desc="Processing cells", total=len(cell_id_list))) as progress_bar:
    fibrosis_intensity_outside_cells = Parallel(n_jobs=-1)(delayed(compute_intensity_outside_cell)(idx) for idx in cell_id_list)

In [ ]:
with open('Y:/coskun-lab/Aditi/01/Results/cell_pixel_barcodes' + key + '/radius100' + '/fibrosis_intensity_counts_outside_cells.pickle', 'wb') as f: 
    pickle.dump(fibrosis_intensity_outside_cells, f)